## Sediment Analysis with Land Masking (SPM & Hue Angle)
analysis of high-sediment areas using SPM and Hue Angle data
land-masking using a coastline shapefile.

In [ ]:
import rasterio
from rasterio import features, mask
from rasterio.mask import mask
from rasterio.merge import merge
import numpy as np
import geopandas as gpd
from shapely.geometry import mapping
import os
from skimage import feature
import matplotlib.pyplot as plt
from matplotlib import colors
from scipy.ndimage import distance_transform_edt

tile1_path = r"../data/Area1"
tile2_path = r"../data/Area1/Tile2"

coast_path = r"../ROI/iho.shp"
#months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

### Merging tiles

In [ ]:
# =========================================================

# =========================================================
# FILE PATHS
# =========================================================
month='Jan'
# --- SPM ---
spm_tile1 = os.path.join(
    tile1_path,
    month,
    'monthly_median_spm_masked.tif'
)

spm_tile2 = os.path.join(
    tile2_path,
    month,
    'monthly_median_spm_masked.tif'
)

# --- HUE ---
hue_tile1 = os.path.join(
    tile1_path,
    month,
    'monthly_median_hue_masked.tif'
)

hue_tile2 = os.path.join(
    tile2_path,
    month,
    'monthly_median_hue_masked.tif'
)

# =========================================================
# LOAD COASTLINE
# =========================================================

coast_gdf = gpd.read_file(coast_path)

# =========================================================
# MERGE SPM TILES
# =========================================================

spm_srcs = [
    rasterio.open(spm_tile1),
    rasterio.open(spm_tile2)
]

spm_mosaic, spm_transform = merge(spm_srcs)

# First band only
spm_data = spm_mosaic[0]

spm_crs = spm_srcs[0].crs
spm_meta = spm_srcs[0].meta.copy()

# =========================================================
# MERGE HUE TILES
# =========================================================

hue_srcs = [
    rasterio.open(hue_tile1),
    rasterio.open(hue_tile2)
]

hue_mosaic, hue_transform = merge(hue_srcs)

hue_data = hue_mosaic[0]

# =========================================================
# OPTIONAL: MASK LAND
# =========================================================

# Convert coastline to same CRS
coast_gdf = coast_gdf.to_crs(spm_crs)

# Create temporary raster metadata
temp_meta = spm_meta.copy()

temp_meta.update({
    "height": spm_data.shape[0],
    "width": spm_data.shape[1],
    "transform": spm_transform
})

# Save temporary merged raster for masking
output_spm = f'merged_{month}_spm.tif'
with rasterio.open(output_spm, "w", **temp_meta) as dst:
    dst.write(spm_data, 1)


print(f"Saved merged SPM raster: {output_spm}")


In [ ]:
# =========================================================
# SAVE MERGED HUE
# =========================================================
output_hue = f'merged_{month}_hue.tif'

temp_meta.update({
    "height": hue_data.shape[0],
    "width": hue_data.shape[1],
    "transform": masked_transform
})

with rasterio.open(output_hue, 'w', **temp_meta) as dst:
    dst.write(hue_data, 1)

print(f"Saved merged hue raster: {output_hue}")
for src in spm_srcs:
    src.close()

for src in hue_srcs:
    src.close()

In [ ]:


def extract_polygons(data, threshold, transform, crs, output_path, condition='>', min_area=5000):
    if condition == '>':
        m = (data > threshold) & (~np.isnan(data))
    else:
        m = (data < threshold) & (~np.isnan(data))
    
    results = (
        {'properties': {'raster_val': v}, 'geometry': s}
        for i, (s, v) 
        in enumerate(features.shapes(m.astype(np.int16), mask=m, transform=transform))
    )
    
    geoms = list(results)
    if not geoms: return 0
    
    gdf = gpd.GeoDataFrame.from_features(geoms, crs=crs)
    gdf['area_m2'] = gdf.geometry.area
    gdf = gdf[gdf['area_m2'] > min_area]
    
    if not gdf.empty:
        gdf.to_file(output_path, driver='GPKG')
        return len(gdf)
    
    return 0


def apply_land_mask(src_path, coast_gdf):
    with rasterio.open(src_path) as src:
        try:
            out_img, _ = mask.mask(src, coast_gdf.to_crs(src.crs).geometry, invert=False)
            data = out_img[0]
            data[data == 0] = np.nan
            return data, src.transform, src.crs
        except:
            return src.read(1), src.transform, src.crs




## Single Month Analysis 
Analyzing SPM and Hue Angle with Land Masking.

In [ ]:

month = 'Apr'
coast_gdf = gpd.read_file(coast_path)


In [ ]:

#spm_path = os.path.join(tile1_path, month, f'merged_{month}_spm.tif')
#hue_path = os.path.join(tile1_path, month, f'merged_{month}_hue.tif')
# -------------------------
# 1.1 SPM Processing
# -------------------------
spm_data, transform, crs = apply_land_mask(spm_path, coast_gdf)


spm_thresh = np.nanpercentile(spm_data, 90)
print(f'SPM Threshold: {spm_thresh:.2f}')
print(np.round(np.nanpercentile(spm_data, [50, 75, 90, 95, 99]),2))


In [ ]:

extract_polygons(
    spm_data, spm_thresh, transform, crs,
    os.path.join(base_path, month, f'high_spm_{month}_masked.shp'),
    condition='>'
)


In [ ]:

# -------------------------
# 1.2 Hue Processing (STATIC)
# -------------------------
hue_data, _, _ = apply_land_mask(hue_path, coast_gdf)

HUE_THRESH = 120
print(f'Static Hue Threshold: {HUE_THRESH}')


In [ ]:

extract_polygons(
    hue_data, HUE_THRESH, transform, crs,
    os.path.join(base_path, month, f'low_hue_{month}_masked.shp'),
    condition='<'
)


In [ ]:
# some analysis for SLH 
slh_path = os.path.join(tile1_path, month, f'monthly_median_slh_masked.tif')

slh_data, _, _ = apply_land_mask(slh_path, coast_gdf)


In [ ]:
print(np.nanmin(slh_data), np.nanmax(slh_data))

In [ ]:

# -------------------------
# 1.3 Visualization
# -------------------------
from scipy.ndimage import median_filter, gaussian_filter
#from matplotlib.colors import LogNorm

# Smooth
spm_smooth = median_filter(spm_data, size=3)

# Scaling
#vmax = np.nanpercentile(spm_data, 98)

# Plot

# --- PLOT ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# SPM plot
im1 = ax1.imshow(spm_smooth, cmap='turbo', vmin=5, vmax=100)
ax1.set_title(f'SPM (thr={spm_thresh:.2f})')

# Hue plot
im2 = ax2.imshow(hue_data, cmap='hsv', vmin=40, vmax=220)
ax2.set_title(f'Hue (thr<{HUE_THRESH})')

# Remove axes
for ax in [ax1, ax2]:
    ax.axis('off')

# --- COLORBARS ---
cbar1 = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
cbar1.set_label("SPM")

cbar2 = fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
cbar2.set_label("Hue")

plt.tight_layout()
plt.show()

In [ ]:

# Valid pixels
valid_mask = (
    ~np.isnan(spm_data) & np.isfinite(spm_data) &
    ~np.isnan(hue_data) & np.isfinite(hue_data)
)
spmthreshp75 = np.nanpercentile(spm_data, 75)
spmthreshp50 = np.nanpercentile(spm_data, 50)
# Individual masks
spm_mask = spm_data > spm_thresh
hue_mask = hue_data < HUE_THRESH
spmp75 = spm_data > spmthreshp75
spmp50 = spm_data> spmthreshp50
hue100 = hue_data < 100
hue80 = hue_data < 80
hue60 = hue_data < 60
# Combined plume mask
plume_mask = valid_mask & spm_mask & hue_mask

In [ ]:
#MAX_DIST = 15000  # meters (coastal zone limit to avoid offshore atm bias)


#land_mask = np.isnan(spm_data)

#pixel_size = abs(transform.a)
#distance_to_land = distance_transform_edt(~land_mask) * pixel_size

#coastal_zone = distance_to_land <= MAX_DIST
#offshore_zone = distance_to_land > MAX_DIST

### SPM classification

In [ ]:

classification = np.zeros_like(spm_data, dtype=np.uint8)

classification[valid_mask] = 1
classification[spmp50] = 2
classification[spmp75] = 3
classification[spm_mask ] = 4

#classification[plume_mask] = 4
#classification[offshore_zone & spm_mask] = 5

from matplotlib.colors import ListedColormap

cmap = ListedColormap([
    'black',      # 0 no data
    'lightblue',  # 1 water
    'lightgreen', 
    'orange',     # 2 high SPM     # 3 low hue
    'red'        # 4 plume        # 5 offshore bias
])


In [ ]:

from rasterio.warp import reproject, Resampling, transform_bounds

# ----------------------------
# 1. Open raster
# ----------------------------
src = rasterio.open(spm_path)

spm = src.read(1)
classification = np.array(classification)

if classification.shape != spm.shape:
    raise ValueError(f"Shape mismatch: {classification.shape} vs {spm.shape}")

# ----------------------------
# 2. FORCE CRS WITHOUT EPSG LOOKUP
# ----------------------------
# Use PROJ string instead of "EPSG:4326"
dst_crs = "+proj=longlat +datum=WGS84 +no_defs"

# ----------------------------
# 3. Compute bounds transform WITHOUT calculate_default_transform
# ----------------------------
dst_bounds = transform_bounds(
    src.crs,
    dst_crs,
    *src.bounds,
    densify_pts=21
)

minx, miny, maxx, maxy = dst_bounds

# define output grid manually
width = src.width
height = src.height

dst_transform = rasterio.transform.from_bounds(
    minx, miny, maxx, maxy,
    width, height
)

# ----------------------------
# 4. Prepare output arrays
# ----------------------------
spm_wgs84 = np.empty((height, width), dtype=spm.dtype)
classification_wgs84 = np.empty((height, width), dtype=classification.dtype)

# ----------------------------
# 5. Reproject SPM (continuous)
# ----------------------------
reproject(
    source=spm,
    destination=spm_wgs84,
    src_transform=src.transform,
    src_crs=src.crs,
    dst_transform=dst_transform,
    dst_crs=dst_crs,
    resampling=Resampling.bilinear
)

# ----------------------------
# 6. Reproject classification (categorical)
# ----------------------------
reproject(
    source=classification,
    destination=classification_wgs84,
    src_transform=src.transform,
    src_crs=src.crs,
    dst_transform=dst_transform,
    dst_crs=dst_crs,
    resampling=Resampling.nearest
)


In [ ]:

plt.figure(figsize=(10,6))
im = plt.imshow(classification_wgs84, cmap=cmap, vmin=0, vmax=4, extent=[minx, maxx, miny, maxy], origin='upper') # vmin to vmax how many ticks

cbar = plt.colorbar(im, ticks=[0,1,2,3,4])
cbar.ax.set_yticklabels([
    'Land/no data',
    'SPM <P50',
    'SPM P50',
    'SPM P75',
    'SPM P90'
    
])

plt.title('SPM Classification December')
plt.xlabel("Longitude")
plt.ylabel("Latitude")
#plt.axis('off')
plt.show()

### Hue angle classification

In [ ]:
from matplotlib.colors import ListedColormap

classification2 = np.zeros_like(hue_data, dtype=np.uint8)

classification2[valid_mask] = 1
classification2[hue_mask] = 2
classification2[hue100] = 3
classification2[hue80] = 4
classification2[hue60] = 5


cmap = ListedColormap([
    'black',      # 0 no data
    'lightblue',  # 1 water
    'yellow',
    'orange',     # 2 high SPM   # 3 low hue
    'red',
    'purple'              # 4 plume        # 5 offshore bias
])


In [ ]:

from rasterio.warp import reproject, Resampling, transform_bounds

# ----------------------------
# 1. Open raster
# ----------------------------
src = rasterio.open(hue_path)

hue = src.read(1)
classification2 = np.array(classification2)

if classification2.shape != hue.shape:
    raise ValueError(f"Shape mismatch: {classification2.shape} vs {hue.shape}")

# ----------------------------
# 2. FORCE CRS WITHOUT EPSG LOOKUP
# ----------------------------
# Use PROJ string instead of "EPSG:4326"
dst_crs = "+proj=longlat +datum=WGS84 +no_defs"

# ----------------------------
# 3. Compute bounds transform WITHOUT calculate_default_transform
# ----------------------------
dst_bounds = transform_bounds(
    src.crs,
    dst_crs,
    *src.bounds,
    densify_pts=21
)

minx, miny, maxx, maxy = dst_bounds

# define output grid manually
width = src.width
height = src.height

dst_transform = rasterio.transform.from_bounds(
    minx, miny, maxx, maxy,
    width, height
)

# ----------------------------
# 4. Prepare output arrays
# ----------------------------
hue_wgs84 = np.empty((height, width), dtype=hue.dtype)
classification2_wgs84 = np.empty((height, width), dtype=classification2.dtype)

# ----------------------------
# 5. Reproject SPM (continuous)
# ----------------------------
reproject(
    source=hue,
    destination=hue_wgs84,
    src_transform=src.transform,
    src_crs=src.crs,
    dst_transform=dst_transform,
    dst_crs=dst_crs,
    resampling=Resampling.bilinear
)

# ----------------------------
# 6. Reproject classification (categorical)
# ----------------------------
reproject(
    source=classification2,
    destination=classification2_wgs84,
    src_transform=src.transform,
    src_crs=src.crs,
    dst_transform=dst_transform,
    dst_crs=dst_crs,
    resampling=Resampling.nearest
)


In [ ]:

plt.figure(figsize=(10,6))
im = plt.imshow(classification2_wgs84, cmap=cmap, vmin=0, vmax=5, extent=[minx, maxx, miny, maxy], origin='upper') # vmin to vmax how many ticks

cbar = plt.colorbar(im, ticks=[0,1,2,3,4,5])
cbar.ax.set_yticklabels([
    'Land/no data',
    'Hue > 120',
    'Hue < 120',
    'Hue < 100',
    'Hue < 80',
    'Hue < 60'
    
])

plt.title('Hue angle classification December')
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()